# 03 - Feature Engineering
Create customer segments and analytical features.

In [4]:
import pandas as pd
import numpy as np

print("Đang tải dữ liệu...")
df = pd.read_csv('../data/processed/master_sales.csv')

# =====================================================================
# 🌟 PHẦN MỚI THÊM: GHÉP NỐI DỮ LIỆU RETURNS (HOÀN TRẢ)
# =====================================================================
print("Đang xử lý dữ liệu Hoàn trả (Returns)...")
returns_df = pd.read_csv('../data/raw/AdventureWorks Returns Data.csv')
returns_df.columns = [col.lower() for col in returns_df.columns] # Chuẩn hóa tên cột

# Bảng Returns không có ID đơn hàng, nên ta gom nhóm theo Sản phẩm và Khu vực
returns_agg = returns_df.groupby(['productkey', 'territorykey'], as_index=False)['returnquantity'].sum()

# Ép kiểu Int64 để Merge chuẩn xác
df['productkey'] = df['productkey'].astype('Int64')
df['territorykey'] = df['territorykey'].astype('Int64')
returns_agg['productkey'] = returns_agg['productkey'].astype('Int64')
returns_agg['territorykey'] = returns_agg['territorykey'].astype('Int64')

# Ghép số lượng trả hàng vào bảng Master
df = df.merge(returns_agg, on=['productkey', 'territorykey'], how='left')


# =====================================================================
# 1. TRÍCH XUẤT ĐẶC TRƯNG THỜI GIAN (Temporal Features)
# =====================================================================
print("1. Đang trích xuất đặc trưng thời gian...")
df['orderdate'] = pd.to_datetime(df['orderdate'], errors='coerce')
df['order_day'] = df['orderdate'].dt.day
df['order_month'] = df['orderdate'].dt.month
df['order_year'] = df['orderdate'].dt.year
df['day_of_week'] = df['orderdate'].dt.dayofweek 
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)


# =====================================================================
# 2. TẠO CÁC CHỈ SỐ KINH DOANH MỚI (Business Features)
# =====================================================================
print("2. Đang tạo các chỉ số kinh doanh...")
df['profit_margin'] = np.where(df['revenue'] > 0, df['profit'] / df['revenue'], 0)

# 🎯 ĐÁNH DẤU NHÃN MỤC TIÊU (TARGET): Nếu SP ở khu vực đó từng bị trả hàng -> Đánh dấu rủi ro 1
df['is_returned'] = np.where(df['returnquantity'].fillna(0) > 0, 1, 0)


# =====================================================================
# 3. XỬ LÝ NGOẠI LỆ (Outliers) VỚI THU NHẬP
# =====================================================================
print("3. Đang xử lý nhiễu (Outliers)...")
if 'annualincome' in df.columns:
    Q1 = df['annualincome'].quantile(0.25)
    Q3 = df['annualincome'].quantile(0.75)
    upper_bound = Q3 + 1.5 * (Q3 - Q1)
    df['annualincome'] = np.where(df['annualincome'] > upper_bound, upper_bound, df['annualincome'])


# =====================================================================
# 4. MÃ HÓA BIẾN PHÂN LOẠI (One-Hot Encoding)
# =====================================================================
print("4. Đang mã hóa biến phân loại (One-Hot Encoding)...")
cat_columns = ['gender', 'maritalstatus', 'educationlevel', 'occupation', 'country']
cat_columns = [col for col in cat_columns if col in df.columns]
df_encoded = pd.get_dummies(df, columns=cat_columns, drop_first=True)
df_encoded.columns = [col.replace(' ', '_').replace('-', '_').lower() for col in df_encoded.columns]


# =====================================================================
# 5. CHUẨN BỊ BẢNG DỮ LIỆU RIÊNG CHO TỪNG MÔ HÌNH
# =====================================================================
print("5. Đang lưu các bộ dữ liệu ML...")
# XGBoost
numeric_cols = df_encoded.select_dtypes(include=['int32', 'int64', 'float32', 'float64', 'uint8', 'Int64']).columns
xgb_features = [col for col in numeric_cols if col not in ['customerkey', 'productkey', 'territorykey', 'salesterritorykey', 'returnquantity']]
df_xgboost = df_encoded[xgb_features]
df_xgboost.to_csv('../data/processed/ml_features_xgboost.csv', index=False)
print(f"  -> Đã lưu dữ liệu XGBoost: {df_xgboost.shape}")

# K-Means
if 'customerkey' in df.columns:
    max_date = df['orderdate'].max()
    df_rfm = df.groupby('customerkey', as_index=False).agg({
        'orderdate': lambda x: (max_date - x.max()).days,
        'ordernumber': 'nunique',
        'revenue': 'sum'
    })
    df_rfm.columns = ['customerkey', 'recency', 'frequency', 'monetary']
    df_rfm.to_csv('../data/processed/ml_features_rfm.csv', index=False)
    print(f"  -> Đã lưu dữ liệu K-Means RFM: {df_rfm.shape}")

print("✅ HOÀN TẤT FEATURE ENGINEERING!")

Đang tải dữ liệu...
Đang xử lý dữ liệu Hoàn trả (Returns)...
1. Đang trích xuất đặc trưng thời gian...
2. Đang tạo các chỉ số kinh doanh...
3. Đang xử lý nhiễu (Outliers)...
4. Đang mã hóa biến phân loại (One-Hot Encoding)...
5. Đang lưu các bộ dữ liệu ML...
  -> Đã lưu dữ liệu XGBoost: (56046, 19)
  -> Đã lưu dữ liệu K-Means RFM: (17416, 4)
✅ HOÀN TẤT FEATURE ENGINEERING!
